In [1]:
import os
from pathlib import Path
print(os.getcwd())
DATA_ROOT = (Path.cwd().parent / "data").resolve()
DATA_ROOT

/home/famo00001/kincf/stats


PosixPath('/home/famo00001/kincf/data')

In [2]:
from kinodata.data import KinodataDocked

In [3]:
DATA_ROOT = (Path.cwd().parent / "data").resolve()

In [ ]:

data = KinodataDocked(root=str(DATA_ROOT))
# data = KinodataDocked()
data

KinodataDocked(119522)

In [4]:
df = data.df.copy()
df.shape

Reading data frame from /home/famo00001/kincf/data/raw/kinodata_docked_v2.sdf.gz...


UndefinedVariableError: name 'activities' is not defined

In [ ]:
ligand_col = 'compound_structures.canonical_smiles'
kinase_col = 'UniprotID'

pairs = df[[ligand_col, kinase_col]].dropna().drop_duplicates()

n_unique_ligands = pairs[ligand_col].nunique()
n_unique_kinases = pairs[kinase_col].nunique()
n_unique_pairs = len(pairs)

summary = pd.DataFrame({
    'metric': [
        'unique molecules',
        'unique kinases',
        'unique molecule-kinase pairs',
    ],
    'value': [n_unique_ligands, n_unique_kinases, n_unique_pairs],
})
summary

From the .sdf directly

In [4]:

import pandas as pd
from rdkit.Chem import PandasTools

sdf_path = Path(DATA_ROOT / "raw/kinodata_docked_v2.sdf.gz")

df = PandasTools.LoadSDF(
    str(sdf_path),
    smilesName="compound_structures.canonical_smiles",
    molColName="molecule",
    embedProps=True,
    removeHs=True,
)

# Handle old/new column naming
ligand_col = (
    "compound_structures.canonical_smiles"
    if "compound_structures.canonical_smiles" in df.columns
    else "compound_structures_canonical_smiles"
)
activity_type_col = (
    "activities.standard_type"
    if "activities.standard_type" in df.columns
    else "activities_standard_type"
)
kinase_col = "UniprotID"  # expected in KinodataDocked raw SDF

# # Optional pIC50 subset
# if activity_type_col in df.columns:
#     df = df[df[activity_col] == "pIC50"]

pairs = df[[ligand_col, kinase_col]].dropna().drop_duplicates()

molecules_per_kinase = (
    pairs.groupby(kinase_col)[ligand_col].nunique().sort_values(ascending=False)
)
kinases_per_molecule = (
    pairs.groupby(ligand_col)[kinase_col].nunique().sort_values(ascending=False)
)

print("Unique kinases:", pairs[kinase_col].nunique())
print("Unique molecules:", pairs[ligand_col].nunique())
print("\nMolecules per kinase (describe):")
print(molecules_per_kinase.describe())
print("\nKinases per molecule (describe):")
print(kinases_per_molecule.describe())

display(molecules_per_kinase.head(20).rename("n_molecules").reset_index())
display(kinases_per_molecule.head(20).rename("n_kinases").reset_index())


Unique kinases: 271
Unique molecules: 98879

Molecules per kinase (describe):
count     271.000000
mean      509.557196
std       896.466349
min         1.000000
25%        37.500000
50%       126.000000
75%       541.500000
max      6263.000000
Name: compound_structures.canonical_smiles, dtype: float64

Kinases per molecule (describe):
count    98879.000000
mean         1.396555
std          1.862357
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max        140.000000
Name: UniprotID, dtype: float64


,UniprotID,n_molecules
0,P35968,6263
1,P00533,5663
2,O60674,5145
3,P15056,3737
4,P11309,3316
5,P23458,3069
6,P42336,3045
7,P43405,3033
8,Q06187,2825
9,P52333,2782


,compound_structures.canonical_smiles,n_kinases
0,COc1cc(N2CCC(N3CC[NH+](C)CC3)CC2)ccc1Nc1ncc(Cl...,140
1,Cc1cnc(Nc2ccc(OCC[NH+]3CCCC3)cc2)nc1Nc1cccc(S(...,129
2,COc1cc(Nc2ncc(F)c(Nc3ccc4c(n3)NC(=O)C(C)(C)O4)...,128
3,C[C@]12O[C@H](C[C@]1(O)CO)n1c3ccccc3c3c4c(c5c6...,114
4,CC(C)n1nc(-c2cc3cc(O)ccc3[nH]2)c2c(N)ncnc21,109
5,C[NH2+][C@@H]1C[C@H]2O[C@@](C)([C@@H]1OC)n1c3c...,104
6,CCN1CC[NH+](Cc2ccc(NC(=O)Nc3ccc(Oc4cc(NC)ncn4)...,101
7,C[NH+](C)CC(=O)N1CCC(c2ccc(NC(=O)c3[nH]c(C#N)c...,95
8,O=C(c1ccc(/C=C/c2n[nH]c3ccccc23)cc1)N1CC[NH2+]CC1,94
9,CSc1cccc(Nc2ncc3cc(-c4c(Cl)cccc4Cl)c(=O)n(C)c3...,76


In [5]:
save_dir = DATA_ROOT / "stats"
save_dir.mkdir(parents=True, exist_ok=True)
save_dir

PosixPath('/home/famo00001/kincf/data/stats')

In [6]:
rows = [
        ("unique_molecules", pairs[ligand_col].nunique()),
        ("unique_kinases", pairs[kinase_col].nunique()),
        ("unique_molecule_kinase_pairs", len(pairs)),
        ("molecules_per_kinase_min", int(molecules_per_kinase.min())),
        ("molecules_per_kinase_p25", float(molecules_per_kinase.quantile(0.25))),
        ("molecules_per_kinase_median", float(molecules_per_kinase.median())),
        ("molecules_per_kinase_mean", float(molecules_per_kinase.mean())),
        ("molecules_per_kinase_p75", float(molecules_per_kinase.quantile(0.75))),
        ("molecules_per_kinase_max", int(molecules_per_kinase.max())),
        ("kinases_per_molecule_min", int(kinases_per_molecule.min())),
        ("kinases_per_molecule_p25", float(kinases_per_molecule.quantile(0.25))),
        ("kinases_per_molecule_median", float(kinases_per_molecule.median())),
        ("kinases_per_molecule_mean", float(kinases_per_molecule.mean())),
        ("kinases_per_molecule_p75", float(kinases_per_molecule.quantile(0.75))),
        ("kinases_per_molecule_max", int(kinases_per_molecule.max())),
    ]
report_df = pd.DataFrame(rows, columns=["metric", "value"])
report_df.to_csv(save_dir / "summary_metrics.csv", index=False)
report_df.head()

,metric,value
0,unique_molecules,98879.0
1,unique_kinases,271.0
2,unique_molecule_kinase_pairs,138090.0
3,molecules_per_kinase_min,1.0
4,molecules_per_kinase_p25,37.5


In [8]:
df.columns

Index(['docking.posit_probability', 'docking.chemgauss_score',
       'activities.activity_id', 'assays.chembl_id',
       'target_dictionary.chembl_id', 'molecule_dictionary.chembl_id',
       'molecule_dictionary.max_phase', 'activities.standard_type',
       'activities.standard_value', 'activities.standard_units',
       'compound_structures.canonical_smiles',
       'compound_structures.standard_inchi', 'component_sequences.sequence',
       'assays.confidence_score', 'docs.chembl_id', 'docs.year',
       'docs.authors', 'UniprotID', 'similar.klifs_structure_id',
       'similar.fp_similarity', 'docking.predicted_rmsd', 'ID', 'molecule'],
      dtype='object')

In [8]:
from plot_dataset_stats import (
    plot_coverage_curve,
    plot_histogram,
    plot_pair_degree_hexbin,
    plot_rank_frequency,
    plot_shared_kinase_heatmap,
)


In [9]:
pairs_for_plot = pairs.rename(columns={ligand_col: "ligand_id", kinase_col: "kinase_id"})

plot_histogram(
    molecules_per_kinase,
    title="Molecules per kinase",
    xlabel="# molecules for kinase",
    save_path=save_dir / "hist_molecules_per_kinase.png",
)

plot_histogram(
    kinases_per_molecule,
    title="Kinases per molecule",
    xlabel="# kinases for molecule",
    save_path=save_dir / "hist_kinases_per_molecule.png",
)

plot_rank_frequency(
    molecules_per_kinase,
    title="Rank-frequency of molecules per kinase",
    ylabel="# molecules",
    save_path=save_dir / "rank_frequency_molecules_per_kinase.png",
)

plot_rank_frequency(
    kinases_per_molecule,
    title="Rank-frequency of kinases per molecule",
    ylabel="# kinases",
    save_path=save_dir / "rank_frequency_kinases_per_molecule.png",
)

plot_coverage_curve(
    molecules_per_kinase,
    title="Coverage by top kinases",
    ylabel="Cumulative fraction of molecule-kinase pairs",
    save_path=save_dir / "coverage_top_kinases.png",
)

plot_coverage_curve(
    kinases_per_molecule,
    title="Coverage by top molecules",
    ylabel="Cumulative fraction of molecule-kinase pairs",
    save_path=save_dir / "coverage_top_molecules.png",
)

shared = plot_shared_kinase_heatmap(
    pairs_for_plot,
    top_k=20,
    save_path=save_dir / "shared_ligands_top20_kinases.png",
)

edge_view = plot_pair_degree_hexbin(
    pairs_for_plot,
    gridsize=45,
    save_path=save_dir / "edge_degree_hexbin.png",
)

display(shared.head())
display(edge_view.head())
print(f"Saved plots to: {save_dir}")


kinase_id,O00329,O14965,O60674,P00533,P04629,P08581,P11309,P11362,P12931,P15056,P23458,P28482,P31749,P35968,P42336,P43405,P52333,Q06187,Q16539,Q9P1W9
kinase_id,,,,,,,,,,,,,,,,,,,,
O00329,2605,12,2,42,2,1,8,2,88,4,6,1,18,60,602,3,2,18,1,3
O14965,12,2275,26,29,29,31,4,21,47,9,16,15,12,114,3,20,15,22,3,4
O60674,2,26,5145,28,66,23,11,20,63,16,2184,6,6,76,3,49,1488,70,8,7
P00533,42,29,28,5663,25,46,14,71,232,48,20,27,10,520,46,17,74,116,20,8
P04629,2,29,66,25,2291,27,14,28,33,17,17,21,5,64,3,23,24,20,5,6


,ligand_id,kinase_id,n_kinases_for_ligand,n_ligands_for_kinase
0,Nc1ncnc2c1c(-c1cccc(Oc3ccccc3)c1)cn2C1CCCC1,P35968,2,6263
1,Nc1ncnc2c1c(-c1cccc(Oc3ccccc3)c1)cn2C1CCCC1,Q02763,2,686
2,CN(c1ccccc1)c1ncnc2ccc(N/N=N/Cc3ccccn3)cc12,P00533,1,5663
3,CC(=C(C#N)C#N)c1ccc(NC(=O)CCC(=O)[O-])cc1,P00533,1,5663
4,O=C([O-])/C=C/c1ccc(O)cc1,P00533,1,5663


Saved plots to: /home/famo00001/kincf/data/stats
